In [1]:
import pandas as pd
import csv
from dotenv import load_dotenv
from groq import Groq

**Pandas**  
When there is a CSV data, efficient and effective way to handle is to use pandas  
This can be used for various data handlind and manipulation  
That makes a good choice for Data Retirieval mechanism

In [2]:
Data = pd.read_csv ('Student_Performance.csv')
print (Data.dtypes)

student_id                        object
age                                int64
gender                            object
study_hours_per_day              float64
social_media_hours               float64
netflix_hours                    float64
part_time_job                     object
attendance_percentage            float64
sleep_hours                      float64
diet_quality                      object
exercise_frequency                 int64
parental_education_level          object
internet_quality                  object
mental_health_rating               int64
extracurricular_participation     object
exam_score                       float64
dtype: object


>Check pandas functions that can be used for data calculations and inference

In [3]:
# various data computation possible
print ('Number of records : ', len (Data))
print ('Min, Avg, Max Study Hours : ', Data['study_hours_per_day'].min(), Data['study_hours_per_day'].mean(), Data['study_hours_per_day'].max())
print ("Values in Paretal Education :", Data['parental_education_level'].unique())
print ("Values in Internet Quality :", Data['internet_quality'].unique())

Number of records :  1000
Min, Avg, Max Study Hours :  0.0 3.5501000000000005 8.3
Values in Paretal Education : ['Master' 'High School' 'Bachelor' nan]
Values in Internet Quality : ['Average' 'Poor' 'Good']


**Filtering**  
Pandas provides conditional filter / slicing functions that can be used as part of retrieval  
Result again in another pandas dataframe

In [4]:
# filter for poor internet quality and score more than 90
# All columns considered
Filtered = Data[(Data['internet_quality'] =='Poor') & (Data['exam_score'] > 90.0)]
Filtered

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
184,S1184,17,Female,4.9,0.0,0.7,No,71.2,6.4,Fair,5,Master,Poor,5,No,96.2
230,S1230,22,Male,4.2,0.3,0.1,No,80.0,7.9,Poor,3,High School,Poor,9,Yes,100.0
315,S1315,24,Female,4.5,3.4,0.4,No,79.6,6.1,Fair,5,High School,Poor,10,No,98.8
320,S1320,24,Female,4.6,1.8,1.5,No,80.9,8.2,Fair,5,NaN,Poor,9,Yes,94.2
336,S1336,21,Female,6.6,2.2,2.7,No,73.0,6.4,Good,2,Bachelor,Poor,8,Yes,100.0
567,S1567,24,Male,4.7,4.1,3.0,No,71.2,4.9,Good,3,Bachelor,Poor,10,Yes,93.1
597,S1597,23,Male,4.9,1.6,4.0,No,87.1,8.0,Fair,5,Bachelor,Poor,7,No,95.5
606,S1606,23,Male,6.8,2.5,0.7,No,97.2,4.4,Good,1,Bachelor,Poor,4,No,93.1
611,S1611,18,Male,6.0,1.6,1.3,No,89.7,5.5,Fair,6,High School,Poor,7,No,100.0
688,S1688,23,Male,5.0,1.8,1.6,No,100.0,5.7,Good,6,High School,Poor,7,No,92.9


In [5]:
# Filter for above average study hours and attendance, but low marks
Filtered = Data[(Data['study_hours_per_day'] >= Data['study_hours_per_day'].mean()) & 
                (Data['attendance_percentage'] >= Data['attendance_percentage'].mean()) & 
                (Data['exam_score'] < 60)]
Filtered

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
179,S1179,19,Female,3.8,1.7,3.8,No,90.2,5.3,Good,2,High School,Average,4,No,58.1
491,S1491,21,Female,4.0,4.3,2.4,No,84.8,8.3,Good,0,High School,Good,2,No,53.0
730,S1730,19,Female,4.3,2.7,2.6,Yes,91.9,5.0,Good,3,Bachelor,Good,1,No,58.4
763,S1763,18,Male,3.9,2.4,0.0,Yes,91.1,5.5,Good,3,High School,Poor,1,Yes,59.5
782,S1782,24,Female,3.9,3.3,1.1,Yes,90.6,8.6,Fair,0,NaN,Good,3,No,56.0
804,S1804,22,Female,4.3,4.0,1.1,Yes,88.4,4.4,Good,2,High School,Average,2,Yes,59.0


**Use as Context**  
The data filtered using pandas used as context to provide LLM the data  
This data retrieval can be quick and effective  


In [6]:
load_dotenv()
client = Groq()
# client = Groq(api_key="Your Key")

>Specific instructions in terms of Response

In [7]:
R_Instr = "Using the context given, provide response to the user question or statement.\
            Context is provided as CSV formatted string.\
            Answer to the question with details"

In [11]:
# User prompt
Prompt = "Why do you think students score low marks despite attending class and studying well?"

Filtered = Data[(Data['study_hours_per_day'] >= Data['study_hours_per_day'].mean()) & 
                (Data['attendance_percentage'] >= Data['attendance_percentage'].mean()) & 
                (Data['exam_score'] < 60)]
#convert to CSV string for llm context because of tabular data. csv is better than plain text
Context = Filtered.to_csv (index=False, float_format='%.1f') 
print('context: ', Context)

# Invoke LLM with prompt and context
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context + "Query : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

print (completion.choices[0].message.content)

context:  student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
S1179,19,Female,3.8,1.7,3.8,No,90.2,5.3,Good,2,High School,Average,4,No,58.1
S1491,21,Female,4.0,4.3,2.4,No,84.8,8.3,Good,0,High School,Good,2,No,53.0
S1730,19,Female,4.3,2.7,2.6,Yes,91.9,5.0,Good,3,Bachelor,Good,1,No,58.4
S1763,18,Male,3.9,2.4,0.0,Yes,91.1,5.5,Good,3,High School,Poor,1,Yes,59.5
S1782,24,Female,3.9,3.3,1.1,Yes,90.6,8.6,Fair,0,,Good,3,No,56.0
S1804,22,Female,4.3,4.0,1.1,Yes,88.4,4.4,Good,2,High School,Average,2,Yes,59.0

**What the data are telling us**

| Student | Attendance % | Study hrs/day | Sleep hrs | Social‑media hrs | Netflix hrs | Part‑time job | Mental‑health rating (1 = poor, 5 = excellent) | Exam score |
|---------|--------------|--------------|-----------|------------------|-------------|----

>Let's try adapting the response

In [12]:
R_Instr = "Using the context given, provide response to the user question or statement.\
            Context is provided as CSV formatted string.\
            Provide comprehensive response"

In [13]:
# User prompt
Prompt = "Students who have good sleep patterns, are they utlising the time properly?"

# Filter relevant rows for above average sleep time and >75 mark. Only relevant columns
Filtered = Data.loc[(Data['sleep_hours'] >= Data['sleep_hours'].mean()*1.1) & 
                (Data['exam_score'] >= 75),
                ['study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 'attendance_percentage']]
Context = Filtered.to_csv (index=False, float_format='%.1f')

# Invoke LLM with prompt and context
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context + "Query : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

print (completion.choices[0].message.content)

**Quick TL‑DR**  
- The data we have does not contain a direct “sleep‑quality” variable, so we have to use **proxy indicators** (high class attendance, high study time, and low leisure‑time screen use) to guess who is likely getting a good night’s sleep.  
- When we look at those “likely‑well‑rested” students, the pattern that emerges is that **they tend to use their time more productively** – they study more, spend less time on Netflix and social media, and usually have higher attendance.  
- The opposite is also true: students who spend a lot of time on Netflix or social media generally have lower study hours and lower attendance, suggesting a less efficient use of their waking hours.

Below is a step‑by‑step walk‑through of the analysis, the numbers that back it up, and a few practical take‑aways.

---

## 1. How We Approximate “Good Sleep Patterns”

| Proxy | Why it works |
|-------|--------------|
| **Attendance ≥ 90 %** | Regular attendance is strongly linked in the literature to

**Get Code from LLM**  
Like in SQL, in pandas also we can take help of LLM to generate code  
This would require specific instructions to be provided for code generation  
The generated code can be used to extract data from dataframe and then use it as context

In [ ]:
C_Instr = "For the given pandas dtypes, write code lines that can filter out data to answer user question.\
            Study the pandas dtypes and user question clearly\
            Give only the python code lines for necessary filtering of the Dataframe variable.\
            Code that can filter and return dataframe. Assign results to 'Result'\
            No additional code / string"

In [ ]:
# Invoke LLM with prompt and context
Prompt = "Who all have good score despite having attendance below 75?"
# Prompt = "how many scored > 90% with less than 70% attendance?"
# Prompt = "What are the students who score good despite of low study time do?"

Data_Frame_Name = "Data"

Dtype = str(Data.dtypes)

messages=[
    {
        "role": "system",
        "content": C_Instr
    },

    {
        "role": "user",
        "content": "Data.dtype : \n"+ Dtype + "Dataframe variable : 'Data'\n Question : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

Code = completion.choices[0].message.content
Code_Clean = Code.strip ("`")
print (Code_Clean)

exec (Code_Clean)
# print (Result)

Context = Result.to_csv (index=False, float_format='%.1f')

# Invoke LLM with prompt and context
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context + "Query : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

print (completion.choices[0].message.content)

**Pandas SQL**  
using the pandasql library, SQL query can be raised on data frame (treatig like a table)  
The psql library can handle basic SQL queries which are typically handled by SQLite  
Since it can be used as SQL queries, it can be integrated in RAG pipe line (query by LLM etc)

In [ ]:
import pandasql as psql

In [ ]:
Query = "SELECT * FROM Data WHERE extracurricular_participation = 'Yes'"

result = psql.sqldf(Query)

print(result)